<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> 一书的补充代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 附录 F：LLM 评估的常见方法

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.0
torch version: 2.7.1
tokenizers version: 0.21.2


&nbsp;
## F.1 理解 LLM 的主要评估方法

- 本节没有代码

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F01_raschka.webp" width="500px">

&nbsp;
### F.2 评估答案选项准确率

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F02_raschka.webp" width="500px">

- 注意，此图描绘的是基于多选题的评估（如 MMLU）的简化版本，其中我们将生成的输出字母与正确答案字母进行比对
- 在实践中，其变体包括对数概率评分，其中不是仅检查最终字母，而是计算模型认为每个候选答案的可能性
- 对于推理模型，这还可以包括评估当正确答案被输入模型时产生的可能性
- 无论哪种情况，评估仍然检查模型是否选择了预定义的答案之一
- （输出概率分数在第 4 章中有更详细的讨论，我们在那里改进了文本生成函数）

&nbsp;
#### F.2.1 加载模型

In [2]:
from pathlib import Path
import torch

from reasoning_from_scratch.ch02 import (
    get_device
)
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)

device = get_device()
torch.set_float32_matmul_precision("high")

# 如果你遇到兼容性问题，请尝试
# 取消注释下面的行并重新运行笔记本
# device = "cpu"

WHICH_MODEL = "base"

if WHICH_MODEL == "base":

    download_qwen3_small(
        kind="base", tokenizer_only=False, out_dir="qwen3"
    )

    tokenizer_path = Path("qwen3") / "tokenizer-base.json"
    model_path = Path("qwen3") / "qwen3-0.6B-base.pth"
    tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

elif WHICH_MODEL == "reasoning":

    download_qwen3_small(
        kind="reasoning", tokenizer_only=False, out_dir="qwen3"
    )

    tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
    model_path = Path("qwen3") / "qwen3-0.6B-reasoning.pth"
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )

else:
    raise ValueError(f"Invalid choice: WHICH_MODEL={WHICH_MODEL}")


model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device)


USE_COMPILE = False  # Set to true to enable compilation
if USE_COMPILE:
  torch._dynamo.config.allow_unspec_int_on_nn_module = True
  model = torch.compile(model)

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date
✓ qwen3/tokenizer-base.json already up-to-date


&nbsp;
#### F.2.2 检查生成的答案字母

In [3]:
example = {
    "question": (
        "How many ways are there to put 4 distinguishable"
        " balls into 2 indistinguishable boxes?"
    ),
    "choices": ["7", "11", "16", "8"],
    "answer": "D",
}

def format_prompt(example):
    return (
        f"{example['question']}\n"
        f"A. {example['choices'][0]}\n"
        f"B. {example['choices'][1]}\n"
        f"C. {example['choices'][2]}\n"
        f"D. {example['choices'][3]}\n"
        "Answer: "  # trailing space encourages a single-letter next token
    )

prompt = format_prompt(example)
print(prompt)

How many ways are there to put 4 distinguishable balls into 2 indistinguishable boxes?
A. 7
B. 11
C. 16
D. 8
Answer: 


---


- 你可以通过 `datasets` 库（可通过 `pip install datasets` 或 `uv add datasets` 安装）直接从 MMLU 数据集加载示例：

```python
from datasets import load_dataset

configs = get_dataset_config_names("cais/mmlu")
dataset = load_dataset("cais/mmlu", "high_school_mathematics")

# 检查测试集中的第一个示例：
example = dataset["test"][0]
print(example)
```

- 上面，我们使用了 `"high_school_mathematics"` 子集；要获取其他子集的列表，使用以下代码：


```python
from datasets import get_dataset_config_names

subsets = get_dataset_config_names("cais/mmlu")
print(subsets)
```

---

In [4]:
prompt_ids = tokenizer.encode(prompt)
prompt_fmt = torch.tensor(prompt_ids, device=device).unsqueeze(0)

- 我们生成几个 token 并提取模型打印的第一个字母 A/B/C/D 的实例：

In [5]:
from reasoning_from_scratch.ch02 import generate_text_basic_stream_cache


def predict_choice(
    model, tokenizer, prompt_fmt, max_new_tokens=8
):
    pred = None
    for t in generate_text_basic_stream_cache(
        model=model,
        token_ids=prompt_fmt,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    ):
        answer = tokenizer.decode(t.squeeze(0).tolist())
        for letter in answer:
            letter = letter.upper()
            if letter in "ABCD":
                pred = letter
                break
        if pred:  # stop as soon as a letter appears
            break
    return pred

In [6]:
pred1 = predict_choice(model, tokenizer, prompt_fmt)

print(
    f"Generated letter: {pred1}\n"
    f"Correct? {pred1 == example['answer']}"
)

Generated letter: C
Correct? False


&nbsp;
### F.3 使用验证器检查答案

- 本节没有代码（参见第 3 章）

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F03_raschka.webp" width="500px">

<br>
&nbsp;

### F.4 使用偏好和排行榜比较模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F04_raschka.webp" width="500px">

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F05_raschka.webp" width="500px">

- Elo 评分（“400 算法”）灵感来自国际象棋排名：https://en.wikipedia.org/wiki/Performance_rating_(chess)
- 注意，LM Arena 已切换到统计 Bradley-Terry 模型，该模型提供类似 Elo 的分数；但是，成对排名的相同概念仍然适用

In [7]:
# 成对“竞技场投票”，其中第一个模型是胜者，
# 第二个模型是败者
votes = [
    ("GPT-5", "Claude-3"),  # First match-up: GPT-5 was preferred over Claude-3
    ("GPT-5", "Llama-4"),
    ("Claude-3", "Llama-3"),
    ("Llama-4", "Llama-3"),
    ("Claude-3", "Llama-3"),
    ("GPT-5", "Llama-3"),
]

In [8]:
def elo_ratings(vote_pairs, k_factor=32, initial_rating=1000):
    # 使用相同的基础评分初始化所有模型
    ratings = {
        model: initial_rating
        for pair in vote_pairs
        for model in pair
    }

    # 每场匹配后更新评分
    for winner, loser in vote_pairs:

        # 根据评分计算当前胜者的期望得分
        expected_winner = 1.0 / (
            1.0 + 10 ** ((ratings[loser] - ratings[winner]) / 400.0)
        )

        # k_factor 决定评分更新的敏感度
        ratings[winner] = (
            ratings[winner] + k_factor * (1 - expected_winner)
        )
        ratings[loser] = (
            ratings[loser] + k_factor * (0 - (1 - expected_winner))
        )

    return ratings

In [9]:
ratings = elo_ratings(votes, k_factor=32, initial_rating=1000)

for model in sorted(ratings, key=ratings.get, reverse=True):
    print(f"{model:8s} : {ratings[model]:.1f}")

GPT-5    : 1043.7
Claude-3 : 1015.2
Llama-4  : 1000.7
Llama-3  : 940.4


- 期望胜者得分的计算方式如下：

$$\text{expected\_winner} \;=\; \frac{1}{1 + 10^{\tfrac{\text{rating\_loser} - \text{rating\_winner}}{400}}}
$$

- 直觉理解：
    - 如果 rating_winner >> rating_loser：
       - 指数 → 非常小的负数
       - 分母 ≈ 1
       - expected_winner ≈ 1（几乎确定获胜）
    - 如果 rating_winner << rating_loser：
       - 指数 → 非常大的正数
       - 分母 → 非常大
       - expected_winner ≈ 0（几乎确定落败）
    - 如果 rating_winner == rating_loser：
       - 指数 = 0
       - 分母 = 2
       - expected_winner = 0.5（旗鼓相当）

&nbsp;
### F.5 使用其他 LLM 评判响应

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F06_raschka.webp" width="500px">

- 在本节中，我们使用另一个更大的 LLM 自动化评估微调 LLM 的响应
- 具体来说，我们使用一个经过指令微调的 200 亿参数 gpt-oss 模型（由 OpenAI 开发），可以通过 ollama（[https://ollama.com](https://ollama.com)）在本地运行

- Ollama 是一个用于高效运行 LLM 的开源应用
- 它是 llama.cpp（[https://github.com/ggerganov/llama.cpp](https://github.com/ggerganov/llama.cpp)）的包装器，后者用纯 C/C++ 实现 LLM 以最大化效率
- 注意，它是一个用于使用 LLM 生成文本（推理）的工具，而不是用于训练或微调 LLM
- 在运行下面的代码之前，请访问 [https://ollama.com](https://ollama.com) 并按照说明安装 ollama（例如，点击“Download”按钮并下载适用于你操作系统的 ollama 应用）

- 对于 macOS 和 Windows 用户，点击你下载的 ollama 应用；如果提示你安装命令行使用，请选择“是”
- Linux 用户可以使用 ollama 网站上提供的安装命令
- 有 3 种方式可以在我们的计算机上运行 ollama：

**1. `ollama serve`**

- 这会将 ollama 后端作为服务器运行，通常在 `http://localhost:11434` 上。在我们通过 API 调用之前，它不会加载模型。如果我们想通过 Python 使用 ollama，这就是我们需要的。

**2. `ollama run gpt-oss:20b`**

- 这是一个便捷包装器。如果服务器尚未运行，它会启动服务器，然后下载模型（第一次），并将我们带入一个交互式终端，在那里我们可以与模型聊天。在幕后，它使用相同的服务器 API。

**3. Ollama 桌面应用**

- 这会自动运行相同的后端，并在其之上提供一个 GUI（如上图所示）。
它还会应用默认设置（系统提示词、温度、停止序列），这可以解释为什么答案看起来与原始 API 使用不同。

---

**注意**：

- 在终端中运行 `ollama serve` 时，如上所述，你可能会遇到错误信息 `Error: listen tcp 127.0.0.1:11434: bind: address already in use`
- 如果是这样，请尝试使用命令 `OLLAMA_HOST=127.0.0.1:11435 ollama serve`（如果该地址也在使用中，请尝试将数字递增一，直到找到未使用的地址）

---

- 例如，要尝试 ollama，我们可以使用 `ollama run gpt-oss:20b` 来试用 200 亿参数的 gpt-oss 20B 模型。该模型（约 13 GB）将在你第一次运行此命令时自动下载。（或者，你可以在桌面应用中使用它，类似于之前的图片。）

```bash
ollama run gpt-oss:20b
```


- 输出如下所示：

```
$ ollama run gpt-oss:20b
pulling manifest 
pulling b112e727c6f1: 100% ▕█████████████████████████████████▏  13 GB                         
pulling fa6710a93d78: 100% ▕█████████████████████████████████▏ 7.2 KB                         
pulling f60356777647: 100% ▕█████████████████████████████████▏  11 KB                         
pulling d8ba2f9a17b3: 100% ▕█████████████████████████████████▏   18 B                         
pulling 55c108d8e936: 100% ▕█████████████████████████████████▏  489 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success
```

- 关于 gpt-oss 的更多信息，请参阅我的深度文章 [From GPT-2 to gpt-oss: Analyzing the Architectural Advances](https://magazine.sebastianraschka.com/p/from-gpt-2-to-gpt-oss-analyzing-the)
- 使用 ollama 和 `"gpt-oss:20b"` 模型（200 亿参数模型）需要 13 GB 的 RAM；如果你的机器不支持，你可以尝试更小的模型，如 40 亿参数的 `qwen3:4b` 模型，只需大约 4 GB 的 RAM
- 或者，如果你的机器支持，你也可以使用更大的 1200 亿参数 gpt-oss（`qwen3:235b`）或甚至 2350 亿参数的 Qwen3 模型（`qwen3:235b`）
- 下载完成后，你将看到一个命令行提示符，允许你与模型聊天
- 尝试一个提示词，如“What is 1+2?”，应该返回类似以下的输出

```
>>> What is 1+2?
Thinking...
User asks: "What is 1+2?" This is simple: answer 3. Provide explanation? Possibly ask for simple 
arithmetic. Provide answer: 3.
...done thinking.

1 + 2 = **3**
```

- 你可以使用输入 `/bye` 来结束这个会话

- 以下代码检查 ollama 会话是否正确运行，然后再使用 ollama 评估我们在上一节中生成的测试集响应

In [10]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError(
        "Ollama not running. Launch ollama before proceeding."
    )
print("Ollama running:", check_if_running("ollama"))

Ollama running: True


- 现在，与我们之前使用的 `ollama run` 命令与模型交互的替代方式是通过 Python 中的 REST API 使用以下函数
- 在运行本笔记本中的下一个单元格之前，请确保 ollama 仍在运行（之前的代码单元格应打印 `"Ollama running: True"`）
- 接下来，运行以下代码单元格来查询模型

In [11]:
import json
import requests


def query_model(
    prompt,
    model="gpt-oss:20b",
    # 如果你使用了 OLLAMA_HOST=127.0.0.1:11435 ollama serve
    # 请将地址从 11434 更新为 11435
    url="http://localhost:11434/api/chat"
):
    # 创建数据负载作为字典
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {     # Settings below are required for deterministic responses
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    # 发送 POST 请求
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data

In [12]:
ollama_model = "gpt-oss:20b"
result = query_model("What is 1+2?", ollama_model)
print(result)

3


- 现在，使用我们上面定义的 `query_model` 函数，我们可以评估我们自己模型的响应

In [16]:
def rubric_prompt(instruction, reference_answer, model_answer):
    rubric = (
        "You are a fair judge assistant. You will be given an instruction, "
        "a reference answer, and a candidate answer to evaluate, according "
        "to the following rubric:\n\n"
        "1: The response fails to address the instruction, providing "
        "irrelevant, incorrect, or excessively verbose content.\n"
        "2: The response partially addresses the instruction but contains "
        "major errors, omissions, or irrelevant details.\n"
        "3: The response addresses the instruction to some degree but is "
        "incomplete, partially correct, or unclear in places.\n"
        "4: The response mostly adheres to the instruction, with only "
        "minor errors, omissions, or lack of clarity.\n"
        "5: The response fully adheres to the instruction, providing a "
        "clear, accurate, and relevant answer in a concise and efficient "
        "manner.\n\n"
        "Now here is the instruction, the reference answer, and the "
        "response.\n"
    )

    prompt = (
        f"{rubric}\n"
        f"Instruction:\n{instruction}\n\n"
        f"Reference Answer:\n{reference_answer}\n\n"
        f"Answer:\n{model_answer}\n\n"
        f"Evaluation: "
    )
    return prompt

- `model_answer` 可以是我们自己模型产生的答案；这里我们为简单起见硬编码了一个可能的模型答案

In [17]:
rendered_prompt = rubric_prompt(
    instruction=(
        "If all birds can fly, and a penguin is a bird, "
        "can a penguin fly?"
    ),
    reference_answer=(
        "Yes, according to the premise that all birds can fly, "
        "a penguin can fly."
    ),
    model_answer=(
        "Yes – under those premises a penguin would be able to fly."
    )
)
print(rendered_prompt)

You are a fair judge assistant. You will be given an instruction, a reference answer, and a candidate answer to evaluate, according to the following rubric:

1: The response fails to address the instruction, providing irrelevant, incorrect, or excessively verbose content.
2: The response partially addresses the instruction but contains major errors, omissions, or irrelevant details.
3: The response addresses the instruction to some degree but is incomplete, partially correct, or unclear in places.
4: The response mostly adheres to the instruction, with only minor errors, omissions, or lack of clarity.
5: The response fully adheres to the instruction, providing a clear, accurate, and relevant answer in a concise and efficient manner.

Now here is the instruction, the reference answer, and the response.

Instruction:
If all birds can fly, and a penguin is a bird, can a penguin fly?

Reference Answer:
Yes, according to the premise that all birds can fly, a penguin can fly.

Answer:
Yes – 

In [18]:
result = query_model(rendered_prompt, ollama_model)
print(result)

**Score: 5**

The candidate answer directly addresses the question, correctly applies the given premises, and concisely states that a penguin would be able to fly. It is accurate, relevant, and clear.
